# Model Training and EvaluationThis notebook trains and compares multiple machine learning models for diabetes risk prediction. The focus is on metrics that matter for healthcare: recall, precision, and ROC-AUC. Threshold tuning is explored to understand the trade-off between false positives and false negatives.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdimport numpy as npimport seaborn as snsimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.pipeline import Pipelinefrom sklearn.compose import ColumnTransformerfrom sklearn.impute import SimpleImputerfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curveimport joblibsys.path.append(str(Path.cwd().parent / 'src'))from data_loader import load_cdc_diabetes_datasns.set_style('whitegrid')plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})MODELS_DIR = Path.cwd().parent / 'models'IMAGES_DIR = Path.cwd().parent / 'images'MODELS_DIR.mkdir(parents=True, exist_ok=True)IMAGES_DIR.mkdir(parents=True, exist_ok=True)

## Load and prepare data

In [ ]:
X, y, df = load_cdc_diabetes_data()X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)print('Train shape:', X_train.shape)print('Test shape:', X_test.shape)print('Class distribution in training set:')print(y_train.value_counts(normalize=True).round(3))

## Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()categorical_features = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()numeric_transformer = Pipeline([    ('imputer', SimpleImputer(strategy='median')),    ('scaler', StandardScaler()),])categorical_transformer = Pipeline([    ('imputer', SimpleImputer(strategy='most_frequent'))])from sklearn.compose import ColumnTransformerpreprocessor = ColumnTransformer(    transformers=[        ('num', numeric_transformer, numeric_features),        ('cat', categorical_transformer, categorical_features),    ],    remainder='passthrough')

## Train modelsMultiple models are trained with the same preprocessing pipeline for fair comparison. Models include logistic regression (interpretable baseline), random forest, and gradient boosting.

In [ ]:
models = {    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),}trained_models = {}for model_name, model in models.items():    print(f"Training {model_name}...")    pipeline = Pipeline([        ('preprocessor', preprocessor),        ('classifier', model),    ])    pipeline.fit(X_train, y_train)    trained_models[model_name] = pipeline    print(f'✓ {model_name} trained')

## Evaluation function

In [ ]:
def evaluate_model(model, X_eval, y_eval, threshold=0.5):    preds_proba = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else None    if preds_proba is not None:        preds = (preds_proba >= threshold).astype(int)    else:        preds = model.predict(X_eval)    results = {        'accuracy': accuracy_score(y_eval, preds),        'precision': precision_score(y_eval, preds, zero_division=0),        'recall': recall_score(y_eval, preds, zero_division=0),        'f1': f1_score(y_eval, preds, zero_division=0),        'roc_auc': roc_auc_score(y_eval, preds_proba) if preds_proba is not None else None,        'confusion_matrix': confusion_matrix(y_eval, preds),        'preds_proba': preds_proba,    }    return results

## Evaluate all models at default threshold (0.5)

In [ ]:
evaluation_results = {}for model_name, model in trained_models.items():    evaluation_results[model_name] = evaluate_model(model, X_test, y_test, threshold=0.5)for model_name, metrics in evaluation_results.items():    print(f'\n### {model_name}')    print(f'Accuracy: {metrics["accuracy"]:.4f}')    print(f'Precision: {metrics["precision"]:.4f}')    print(f'Recall: {metrics["recall"]:.4f}')    print(f'F1-score: {metrics["f1"]:.4f}')    print(f'ROC-AUC: {metrics["roc_auc"]:.4f}')    print(f'Confusion matrix:\n{metrics["confusion_matrix"]}')

## ROC curves comparison

In [ ]:
plt.figure(figsize=(10, 8))for model_name, model in trained_models.items():    if hasattr(model, "predict_proba"):        y_proba = model.predict_proba(X_test)[:, 1]        fpr, tpr, _ = roc_curve(y_test, y_proba)        auc = roc_auc_score(y_test, y_proba)        plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})", linewidth=2)plt.plot([0, 1], [0, 1], "k--", label="Random", linewidth=1)plt.xlabel("False Positive Rate")plt.ylabel("True Positive Rate")plt.title("ROC Curves Comparison")plt.legend(loc="lower right")plt.grid(alpha=0.3)plt.tight_layout()plt.savefig(IMAGES_DIR / "roc_curves.png")plt.show()print("Saved ROC curves plot")

## Precision-recall curves comparison

In [ ]:
plt.figure(figsize=(10, 8))for model_name, model in trained_models.items():    if hasattr(model, "predict_proba"):        y_proba = model.predict_proba(X_test)[:, 1]        precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)        plt.plot(recall_vals, precision_vals, label=f"{model_name}", linewidth=2)plt.xlabel("Recall")plt.ylabel("Precision")plt.title("Precision-Recall Curves")plt.legend()plt.grid(alpha=0.3)plt.xlim([0, 1])plt.ylim([0, 1])plt.tight_layout()plt.savefig(IMAGES_DIR / "precision_recall_curves.png")plt.show()print("Saved precision-recall curves plot")

## Threshold tuning for best modelIn healthcare risk prediction, missing true positives (low recall) can be costly. This section explores using a lower decision threshold to improve recall at the cost of more false positives.

In [ ]:
best_model = trained_models["Gradient Boosting"]y_proba_best = best_model.predict_proba(X_test)[:, 1]thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]threshold_results = []for thresh in thresholds_to_test:    y_pred_thresh = (y_proba_best >= thresh).astype(int)    threshold_results.append({        "threshold": thresh,        "accuracy": accuracy_score(y_test, y_pred_thresh),        "precision": precision_score(y_test, y_pred_thresh, zero_division=0),        "recall": recall_score(y_test, y_pred_thresh, zero_division=0),        "f1": f1_score(y_test, y_pred_thresh, zero_division=0),    })threshold_df = pd.DataFrame(threshold_results)print("Threshold Analysis:")print(threshold_df.round(4))

## Trade-off visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(threshold_df["threshold"], threshold_df["precision"], "o-", label="Precision", linewidth=2, markersize=8)axes[0].plot(threshold_df["threshold"], threshold_df["recall"], "s-", label="Recall", linewidth=2, markersize=8)axes[0].set_xlabel("Threshold")axes[0].set_ylabel("Score")axes[0].set_title("Precision vs Recall by Threshold")axes[0].legend()axes[0].grid(alpha=0.3)axes[0].set_ylim([0, 1])axes[1].plot(threshold_df["threshold"], threshold_df["accuracy"], "o-", label="Accuracy", linewidth=2, markersize=8)axes[1].plot(threshold_df["threshold"], threshold_df["f1"], "s-", label="F1-score", linewidth=2, markersize=8)axes[1].set_xlabel("Threshold")axes[1].set_ylabel("Score")axes[1].set_title("Accuracy vs F1-score by Threshold")axes[1].legend()axes[1].grid(alpha=0.3)axes[1].set_ylim([0, 1])plt.tight_layout()plt.savefig(IMAGES_DIR / "threshold_tradeoff.png")plt.show()print("Saved threshold trade-off plot")

## Understanding the trade-off**Lower threshold (e.g., 0.3)**:- Higher recall: catches more true positives (fewer missed cases)- Lower precision: more false positives (unnecessary interventions)- Better for healthcare risk screening where missing cases is costly**Higher threshold (e.g., 0.7)**:- Lower recall: may miss true positives- Higher precision: fewer false alarms- Better when false positives are very costlyFor diabetes risk prediction, a threshold around 0.4-0.5 may be appropriate to balance catching at-risk individuals while minimizing unnecessary follow-ups.

## Save the best model

In [ ]:
best_model_name = "Gradient Boosting"best_model_path = MODELS_DIR / "best_model.joblib"joblib.dump(trained_models[best_model_name], best_model_path)print(f"Saved best model ({best_model_name}) to {best_model_path}")

## SummaryThis notebook trained and compared three models for diabetes risk prediction. The gradient boosting model showed the strongest overall performance. Threshold tuning demonstrates how to adjust the model for different practical requirements in healthcare contexts.